## Packages


In [44]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

In [45]:

# Download latest version
path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")

print("Path to dataset files:", path)

Path to dataset files: /home/codespace/.cache/kagglehub/datasets/faviovaz/marketing-ab-testing/versions/1


In [46]:

# List files in the dataset directory
files = os.listdir(path)
print("Files in dataset:", files)

# Load the main dataset (adjust filename as needed)
df = pd.read_csv(os.path.join(path, files[0]))
print(df.head())

Files in dataset: ['marketing_AB.csv']
   Unnamed: 0  user id test group  converted  total ads most ads day  \
0           0  1069124         ad      False        130       Monday   
1           1  1119715         ad      False         93      Tuesday   
2           2  1144181         ad      False         21      Tuesday   
3           3  1435133         ad      False        355      Tuesday   
4           4  1015700         ad      False        276       Friday   

   most ads hour  
0             20  
1             22  
2             18  
3             10  
4             14  


## Preprocesing

In [47]:
#Columns
print("Columns in dataset:", df.columns)

Columns in dataset: Index(['Unnamed: 0', 'user id', 'test group', 'converted', 'total ads',
       'most ads day', 'most ads hour'],
      dtype='str')


In [48]:
#Column name cleanup
df.columns = df.columns.str.strip()  # Remove leading/trailing whitespace from column names

In [49]:
#Change white space in columns to _
df.columns = df.columns.str.replace(' ', '_')

In [50]:
#Duplicate check
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [51]:
#Duplicated user ids
print("Duplicated user IDs:", df['user_id'].duplicated().sum())

Duplicated user IDs: 0


In [52]:
# Total users in each group
group_counts = df['test_group'].value_counts()
print("Users in each group:\n", group_counts)

Users in each group:
 test_group
ad     564577
psa     23524
Name: count, dtype: int64


In [53]:
#Total adds per day
daily_adds = df.groupby(['test_group','most_ads_day'])['total_ads'].sum().sort_values(ascending=False)
print("Total adds to cart per day:\n", daily_adds)

Total adds to cart per day:
 test_group  most_ads_day
ad          Friday          2369546
            Monday          2121848
            Sunday          2006360
            Saturday        1980043
            Wednesday       1904940
            Thursday        1839933
            Tuesday         1792031
psa         Thursday         104240
            Friday            94950
            Monday            83582
            Wednesday         80478
            Saturday          80048
            Sunday            77493
            Tuesday           61690
Name: total_ads, dtype: int64


In [54]:
#Null checks
print("Null values in each column:\n", df.isnull().sum())

Null values in each column:
 Unnamed:_0       0
user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64


## A/B Testing

In [55]:
df.columns

Index(['Unnamed:_0', 'user_id', 'test_group', 'converted', 'total_ads',
       'most_ads_day', 'most_ads_hour'],
      dtype='str')

In [56]:
# Summary Statistics
summary_stats = df.groupby('test_group')['total_ads'].agg(['mean', 'median', 'std', 'count'])
print("Summary statistics for total ads by test group:\n", summary_stats)

Summary statistics for total ads by test group:
                  mean  median        std   count
test_group                                      
ad          24.823365    13.0  43.750456  564577
psa         24.761138    12.0  42.860720   23524


In [60]:
df

,Unnamed:_0,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...,...
588096,588096,1278437,ad,False,1,Tuesday,23
588097,588097,1327975,ad,False,1,Tuesday,23
588098,588098,1038442,ad,False,3,Tuesday,23
588099,588099,1496395,ad,False,1,Tuesday,23


# T-Test

In [ ]:
#T-test for total ads
ad_group = df[df['test_group'] == 'ad']['total_ads']
psa_group = df[df['test_group'] == 'psa']['total_ads']
t_stat, p_value = stats.ttest_ind(ad_group, psa_group, equal_var=False)
print("T-test p-value for total ads difference:", p_value)


# Chi-Square

In [84]:
# A/B test for conversion rates using a contingency table
control_successes = df[(df['test_group'] == 'ad') & (df['converted'] == True)].shape[0]
control_trials = df[df['test_group'] == 'ad'].shape[0]
treatment_successes = df[(df['test_group'] == 'psa') & (df['converted'] == True)].shape[0]
treatment_trials = df[df['test_group'] == 'psa'].shape[0]
contingency_table = [
    [control_successes, control_trials - control_successes],
    [treatment_successes, treatment_trials - treatment_successes]
]
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print("Chi-square test p-value for conversion difference:", p_value)
#What does the p value mean in this context?
print("The p-value indicates the probability of observing a difference in conversion rates as extreme as the one observed, assuming the null hypothesis (no difference) is true. A small p-value (typically < 0.05) suggests that the observed difference is statistically significant.")
#dof stands for degrees of freedom, which is a parameter used in various statistical tests to determine the shape of the distribution. In the context of a chi-square test, it is calculated based on the number of categories in the contingency table. For a 2x2 table, the degrees of freedom would be (rows - 1) * (columns - 1) = (2-1)*(2-1) = 1.
print("Degrees of freedom for the chi-square test:", dof)
#expected represents the expected frequencies in each cell of the contingency table under the null hypothesis. It is calculated based on the marginal totals of the table and is used to compare against the observed frequencies to determine if there is a significant difference.
print("Expected frequencies under the null hypothesis:\n", expected)

Chi-square test p-value for conversion difference: 1.9989623063390075e-13
The p-value indicates the probability of observing a difference in conversion rates as extreme as the one observed, assuming the null hypothesis (no difference) is true. A small p-value (typically < 0.05) suggests that the observed difference is statistically significant.
Degrees of freedom for the chi-square test: 1
Expected frequencies under the null hypothesis:
 [[ 14249.28100955 550327.71899045]
 [   593.71899045  22930.28100955]]


In [85]:
#Is the null hypothesis rejected or not?
if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant difference in conversion rates between the ad and PSA groups.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in conversion rates between the ad and PSA groups.")
    

Reject the null hypothesis: There is a significant difference in conversion rates between the ad and PSA groups.


# Bayesian Test

In [86]:
#Bayesian Test for conversion rates
from scipy.stats import beta
# Calculate alpha and beta parameters for control and treatment groups
control_alpha = control_successes + 1
control_beta = control_trials - control_successes + 1
psa_alpha = treatment_successes + 1
psa_beta = treatment_trials - treatment_successes + 1
# Sample from the posterior distributions
control_samples = beta.rvs(control_alpha, control_beta, size=10000)
psa_samples = beta.rvs(psa_alpha, psa_beta, size=10000)
# Calculate the probability that psa is better than control
prob_treatment_better = np.mean(control_samples > psa_samples)
print("Probability that ads are better than psa:", prob_treatment_better)

Probability that ads are better than psa: 1.0


In [87]:
# Confidence Interval for difference in conversion rates
control_rate = control_successes / control_trials
psa_rate = treatment_successes / treatment_trials
diff = control_rate - psa_rate
se_diff = np.sqrt((control_rate * (1 - control_rate) / control_trials) + (psa_rate * (1 - psa_rate) / treatment_trials))
z_score = stats.norm.ppf(0.975)  # 975% confidence
ci_lower = diff - z_score * se_diff
ci_upper = diff + z_score * se_diff
print(f"95% Confidence Interval for difference in conversion rates: [{ci_lower:.4f}, {ci_upper:.4f}]")
#Explain the confidence interval
print("The confidence interval indicates that we are 95% confident that the true difference in conversion rates between the PSA and ad groups lies within this range. If the interval does not include zero, it suggests a statistically significant difference between the groups.")

95% Confidence Interval for difference in conversion rates: [0.0060, 0.0094]
The confidence interval indicates that we are 95% confident that the true difference in conversion rates between the PSA and ad groups lies within this range. If the interval does not include zero, it suggests a statistically significant difference between the groups.


In [89]:
#How much success can be attributed to the ads?
attributable_success = diff * control_trials
print(f"Estimated number of conversions attributable to the ads: {attributable_success:.0f}")

Estimated number of conversions attributable to the ads: 4343
